In [1]:
!pip install tensorrt huggingface_hub ultralytics -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 109.6 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
d

In [2]:
import os
import time
import torch
import numpy as np
import yaml
from pathlib import Path
from huggingface_hub import hf_hub_download, HfApi
from ultralytics import YOLO
from kaggle_secrets import UserSecretsClient

# ================= CONFIGURATION =================
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
REPO_ID = user_secrets.get_secret("REPO_ID")
MODEL_FILE = "models/yolo26n_final_model_best.pt"

DATASET_PATH = Path("/kaggle/input/datasets/hannusiaknysh/uav-detection-dataset/extracted_dataset")
WORKING_DIR = Path("/kaggle/working")
OUTPUT_DIR = Path("/kaggle/working/quantization_results")
OUTPUT_DIR.mkdir(exist_ok=True)

N_WARMUP    = 10   # warmup iterations
N_BENCHMARK = 100  # benchmark iterations
IMGSZ       = 1280
# =================================================

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
def safe_symlink(src: Path, dst: Path):
    """Creates a symlink, unlinking any existing symlink or file at destination first."""
    if dst.is_symlink() or dst.exists():
        try:
            dst.unlink()
        except OSError:
            return
    try:
        os.symlink(src, dst)
    except FileExistsError:
        pass

def prepare_yolo_dataset(dataset_path: Path, working_dir: Path, run_name: str, use_negatives: bool) -> Path:
    """Prepares isolated directories and creates symlinks for images and labels."""
    print(f"\n[DATA PREPARATION] Setting up dataset directories for run: {run_name}")
    
    run_data_dir = working_dir / f"yolo_dataset_{run_name}"
    for split in ["train", "val", "test"]:
        (run_data_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (run_data_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        
    def process_split(split_name: str):
        pos_dir = dataset_path / split_name / "RAW_POSITIVES"
        linked_pos_count = 0
        if pos_dir.exists():
            for jpg_path in pos_dir.glob("*.jpg"):
                txt_path = jpg_path.with_suffix(".txt")
                if txt_path.exists():
                    dest_img = run_data_dir / "images" / split_name / jpg_path.name
                    dest_lbl = run_data_dir / "labels" / split_name / txt_path.name
                    
                    safe_symlink(jpg_path, dest_img)
                    safe_symlink(txt_path, dest_lbl)
                    linked_pos_count += 1

        linked_neg_count = 0
        if use_negatives:
            neg_dir = dataset_path / split_name / "RAW_NEGATIVES"
            if neg_dir.exists():
                for jpg_path in neg_dir.glob("*.jpg"):
                    dest_img = run_data_dir / "images" / split_name / jpg_path.name
                    safe_symlink(jpg_path, dest_img)
                    linked_neg_count += 1

        print(f"[{split_name.upper()}] Symlinked {linked_pos_count:,} positives and {linked_neg_count:,} negatives.")

    process_split("train")
    process_split("val")
    process_split("test")

    yaml_data = {
        "path": str(run_data_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "drone"}
    }
    
    yaml_path = working_dir / f"dataset_{run_name}.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(yaml_data, f, default_flow_style=False)
        
    print(f"[DATA PREPARATION] Created YAML configuration at: {yaml_path}")
    return yaml_path

# Execute dataset prep
DATASET_YAML = prepare_yolo_dataset(DATASET_PATH, WORKING_DIR, run_name="quantization", use_negatives=True)


[DATA PREPARATION] Setting up dataset directories for run: quantization
[TRAIN] Symlinked 8,177 positives and 810 negatives.
[VAL] Symlinked 1,542 positives and 147 negatives.
[TEST] Symlinked 2,281 positives and 243 negatives.
[DATA PREPARATION] Created YAML configuration at: /kaggle/working/dataset_quantization.yaml


In [4]:
def download_weights() -> Path:
    print("\nDownloading weights from HuggingFace...")
    local = hf_hub_download(
        repo_id=REPO_ID,
        filename=MODEL_FILE,
        token=HF_TOKEN,
        repo_type="dataset",
        local_dir="/kaggle/working/hf_weights",
    )
    print(f"  Saved to: {local}")
    return Path(local)

def get_file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 * 1024)

def benchmark_model(model: YOLO, label: str, dummy_img: np.ndarray) -> dict:
    """Measures FPS and latency of the model on a dummy image."""
    print(f"\nBenchmark: {label}")

    # GPU Warmup
    for _ in range(N_WARMUP):
        model(dummy_img, verbose=False, device="cuda")
    torch.cuda.synchronize()

    # Measurement
    times = []
    for _ in range(N_BENCHMARK):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(dummy_img, verbose=False, device="cuda")
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    times = np.array(times) * 1000  # convert to milliseconds
    fps   = 1000 / np.mean(times)

    result = {
        "label":      label,
        "mean_ms":    np.mean(times),
        "fps":        fps,
    }

    print(f"  FPS:         {fps:.1f}")
    print(f"  Latency:     {np.mean(times):.2f} ± {np.std(times):.2f} ms")

    return result

def validate_model_accuracy(model: YOLO, label: str) -> float:
    """Validates the model on the test split and returns mAP@50."""
    print(f"\nValidating accuracy for: {label}...")
    
    # CRITICAL FIX: rect=False prevents bounding box scaling corruption with static TRT engines
    metrics = model.val(
        data=str(DATASET_YAML), 
        split="test", 
        device="cuda", 
        rect=False,
        imgsz=IMGSZ,
        verbose=False
    )
    
    map50 = metrics.box.map50
    print(f"  [ACCURACY] mAP@50 for {label}: {map50:.4f}")
    return map50

In [9]:
def export_and_benchmark():
    weights_path = download_weights()
    dummy_img    = np.random.randint(0, 255, (IMGSZ, IMGSZ, 3), dtype=np.uint8)

    results  = []
    sizes    = {}
    accuracies = {}
    trt_fp16_path = None

    # -------------------------------------------------
    # 1. Baseline PyTorch Model (FP32)
    # -------------------------------------------------
    print("\n" + "="*50)
    print("1. FP32 PyTorch (baseline)")
    print("="*50)
    model_fp32 = YOLO(str(weights_path)).to("cuda")
    sizes["FP32 PyTorch"] = get_file_size_mb(weights_path)
    print(f"  Size: {sizes['FP32 PyTorch']:.1f} MB")
    
    results.append(benchmark_model(model_fp32, "FP32 PyTorch", dummy_img))
    accuracies["FP32 PyTorch"] = validate_model_accuracy(model_fp32, "FP32 PyTorch")

    # -------------------------------------------------
    # 2. TensorRT FP16 (Standard Native Export)
    # -------------------------------------------------
    print("\n" + "="*50)
    print("2. TensorRT FP16 Export")
    print("="*50)
    try:
        print("  Exporting PyTorch to TensorRT FP16...")
        # Clean export, relies on rect=False during validation for accuracy
        engine_path_str = model_fp32.export(
            format="engine",
            imgsz=IMGSZ,
            half=True,       
            workspace=4,
            dynamic=False
        )
        
        trt_fp16_path = Path(engine_path_str)
        sizes["TRT FP16"] = get_file_size_mb(trt_fp16_path)
        print(f"  Size: {sizes['TRT FP16']:.1f} MB")
        
        # Load the compiled engine for benchmarking and validation
        model_trt_fp16 = YOLO(str(trt_fp16_path), task="detect")
        results.append(benchmark_model(model_trt_fp16, "TRT FP16", dummy_img))
        accuracies["TRT FP16"] = validate_model_accuracy(model_trt_fp16, "TRT FP16")

    except Exception as e:
        print(f"  TRT FP16 export failed: {e}")

    # -------------------------------------------------
    # 3. Relative Metrics Calculation and Table Output
    # -------------------------------------------------
    print("\n" + "="*80)
    print("SUMMARY TABLE OF EMPIRICAL RESULTS (KAGGLE TESLA T4)")
    print("="*80)
    print(f"{'Format':<18} {'Size (MB)':<12} {'FPS':<10} {'Latency (ms)':<14} {'mAP@50':<10}")
    print("-" * 80)

    for r in results:
        label = r["label"]
        size = sizes.get(label, 0)
        map50 = accuracies.get(label, 0.0)
        print(f"{label:<18} {size:<12.1f} {r['fps']:<10.1f} {r['mean_ms']:<14.2f} {map50:<10.4f}")

    # -------------------------------------------------
    # 4. Publication on Hugging Face
    # -------------------------------------------------
    print("\n" + "=" * 50)
    print("4. Publication on Hugging Face")
    print("=" * 50)
    try:
        api = HfApi()
        if trt_fp16_path and trt_fp16_path.exists():
            print("  Uploading .engine...")
            api.upload_file(
                path_or_fileobj=str(trt_fp16_path),
                path_in_repo=f"models/{trt_fp16_path.name}",
                repo_id=REPO_ID, 
                token=HF_TOKEN, 
                repo_type="dataset"
            )
            print("  ✅ Successfully saved to HF!")
        else:
            print("  ❌ Engine file not found. Skipping HF upload.")
    except Exception as e:
        print(f"  ❌ Upload to HF failed: {e}")

if __name__ == "__main__":
    export_and_benchmark()


  Saved to: /kaggle/working/hf_weights/models/yolo26n_final_model_best.pt

1. FP32 PyTorch (baseline)
  Size: 5.2 MB

Benchmark: FP32 PyTorch
  FPS:         55.8
  Latency:     17.93 ± 0.55 ms

Validating accuracy for: FP32 PyTorch...
Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 370.0±70.4 MB/s, size: 203.8 KB)
val: Scanning /kaggle/working/yolo_dataset_quantization/labels/test.cache... 2281 images, 243 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2524/2524 481.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 158/158 3.1it/s 50.5s0.3ss
                   all       2524       2281      0.918      0.808      0.888      0.689
Speed: 3.6ms preprocess, 13.5ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /kaggle/working/runs/detect/val-3
  [ACCURACY] mAP@50 for FP32 PyTorch: 0.8875

2. TensorRT FP16 Export
  Export

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 18 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 1.6s, saved as '/kaggle/working/hf_weights/models/yolo26n_final_model_best.onnx' (9.8 MB)

TensorRT: starting export with TensorRT 11.1.0.106...
[06/20/2026-22:43:40] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
TensorRT: converting ONNX to FP16 mixed precision with ModelOpt AutoCast...
2026-06-20 22:43:40,644 - [modelopt][onnx] - WARNING - Shared constants were detected and duplicated accordingly.
2026-06-20 22:43:40,775 - [modelopt][onnx] - INFO - Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']
2026-06-20 22:43:40,776 - [modelopt][onnx] - INFO - Running ONNX Runtime to obtain reference outputs (this may take a while)...
2026-06-20 22:43:41,775 - [modelopt][onnx] - INFO - Skipping node /m

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  ✅ Successfully saved to HF!


In [7]:
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
from pathlib import Path
import cv2
import time
import torch
import numpy as np

# 1. Download the quantized engine from your Hugging Face repo
print("Downloading your quantized engine from HF...")
engine_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="models/yolo26n_final_model_best.engine",
    token=HF_TOKEN,
    repo_type="dataset",
    local_dir="/kaggle/working/hf_weights"
)

# 2. Load the model
model = YOLO(engine_path, task="detect")
print("Model loaded successfully.")

# 3. Load a real image from your test set (the same one used in the previous step)
test_image_path = list(Path("/kaggle/working/yolo_dataset_quantization/images/test").glob("*.jpg"))[0]
real_img = cv2.imread(str(test_image_path))
real_img = cv2.cvtColor(real_img, cv2.COLOR_BGR2RGB)
real_img = cv2.resize(real_img, (1280, 1280)) # Matching IMGSZ

# 4. Benchmark loop (honest latency measurement)
print(f"Running benchmark on real image: {test_image_path.name}")

# GPU Warmup
for _ in range(10):
    model(real_img, verbose=False, device="cuda")
torch.cuda.synchronize()

# Measurement
latencies = []
for _ in range(100):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    
    model(real_img, verbose=False, device="cuda")
    
    torch.cuda.synchronize()
    latencies.append((time.perf_counter() - t0) * 1000)

avg_lat = np.mean(latencies)
print(f"\n{'='*50}")
print(f"RESULTS FOR QUANTIZED ENGINE (1280x1280)")
print(f"{'='*50}")
print(f"Average Latency: {avg_lat:.2f} ms")
print(f"Estimated FPS:   {1000/avg_lat:.1f}")
print(f"{'='*50}")

Model loaded successfully.
Running benchmark on real image: delta_thermal_field_126_f0022.jpg
Loading /kaggle/working/hf_weights/models/yolo26n_final_model_best.engine for TensorRT inference...
[06/21/2026-12:57:26] [TRT] [I] Loaded engine size: 6 MiB
[06/21/2026-12:57:26] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +52, now: CPU 0, GPU 60 (MiB)

RESULTS FOR QUANTIZED ENGINE (1280x1280)
Average Latency: 10.19 ms
Estimated FPS:   98.2


In [6]:
# 2. Load the model
model = YOLO(engine_path, task="detect")
print("Model loaded successfully.")

# 3. Load a real image from your test set (the same one used in the previous step)
test_image_path = list(Path("/kaggle/working/yolo_dataset_quantization/images/test").glob("*.jpg"))[0]
real_img = cv2.imread(str(test_image_path))
# real_img = cv2.cvtColor(real_img, cv2.COLOR_BGR2RGB)
# real_img = cv2.resize(real_img, (1280, 1280)) # Matching IMGSZ

# 4. Benchmark loop (honest latency measurement)
print(f"Running benchmark on real image: {test_image_path.name}")

# GPU Warmup
for _ in range(10):
    model(real_img, verbose=False, device="cuda")
torch.cuda.synchronize()

# Measurement
latencies = []
for _ in range(100):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    
    model(real_img, verbose=False, device="cuda")
    
    torch.cuda.synchronize()
    latencies.append((time.perf_counter() - t0) * 1000)

avg_lat = np.mean(latencies)
print(f"\n{'='*50}")
print(f"RESULTS FOR QUANTIZED ENGINE (1280x1280)")
print(f"{'='*50}")
print(f"Average Latency: {avg_lat:.2f} ms")
print(f"Estimated FPS:   {1000/avg_lat:.1f}")
print(f"{'='*50}")

Model loaded successfully.
Running benchmark on real image: delta_thermal_field_126_f0022.jpg
Loading /kaggle/working/hf_weights/models/yolo26n_final_model_best.engine for TensorRT inference...
[06/21/2026-12:56:00] [TRT] [I] Loaded engine size: 6 MiB
[06/21/2026-12:56:00] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +52, now: CPU 0, GPU 60 (MiB)

RESULTS FOR QUANTIZED ENGINE (1280x1280)
Average Latency: 12.57 ms
Estimated FPS:   79.6
